下面这个 Jupyter 代码会扫描 raw 文件夹下所有检查文件夹，也就是文件夹名前 8 位是合法日期 YYYYMMDD的文件夹。
然后只删除这些检查文件夹直接下面的：
images    
with_data_images    
without_data_images

标注结果.csv

report.txt

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import os
import stat


# =========================
# 1. 修改成你的 raw 根目录
# =========================
RAW_ROOT = Path(r"G:\wxy_codes\Fetal_hydronephrosis\dataset\2第二批数据及处理过程\batch2data_5\raw")


# =========================
# 2. 是否只预览
# True：只打印将要删除的内容，不真正删除
# False：真正删除
# =========================
DRY_RUN = False


# =========================
# 3. 是否删除指定文件夹
# =========================
DELETE_DIRS = True


# =========================
# 4. 是否删除指定文件
# =========================
DELETE_FILES = True


# =========================
# 5. 需要删除的检查文件夹下的子文件夹名字
# 如果暂时不想删除文件夹，可以写成空集合：set()
# =========================
TARGET_DIR_NAMES = {
    "images",
    "with_data_images",
    "without_data_images",
}


# =========================
# 6. 需要删除的检查文件夹下的文件名字
# 如果暂时不想删除文件，可以写成空集合：set()
# =========================
TARGET_FILE_NAMES = {
    "标注结果.csv",
    "report.txt",
}


def is_exam_folder(folder: Path) -> bool:
    """
    判断是否为检查文件夹：
    文件夹名前 8 位是合法日期 YYYYMMDD。
    例如：202412300188 -> 前 8 位是 20241230
    """
    if not folder.is_dir():
        return False

    name = folder.name

    if len(name) < 8:
        return False

    date_part = name[:8]

    if not date_part.isdigit():
        return False

    try:
        datetime.strptime(date_part, "%Y%m%d")
        return True
    except ValueError:
        return False


def handle_remove_readonly(func, path, exc_info):
    """
    处理 Windows 下只读文件或只读文件夹导致无法删除的问题。
    """
    os.chmod(path, stat.S_IWRITE)
    func(path)


def remove_readonly_file(file_path: Path):
    """
    删除只读文件。
    """
    try:
        file_path.unlink()
    except PermissionError:
        os.chmod(file_path, stat.S_IWRITE)
        file_path.unlink()


def collect_targets(raw_root: Path):
    """
    收集所有检查文件夹下需要删除的目标文件夹和目标文件。
    这里只检查“检查文件夹的直接子级”，不会删除更深层级的同名文件或文件夹。
    """
    exam_folders = []
    target_dirs = []
    target_files = []

    for folder in raw_root.rglob("*"):
        if is_exam_folder(folder):
            exam_folders.append(folder)

            # 查找检查文件夹直接子文件夹
            if DELETE_DIRS:
                for dir_name in TARGET_DIR_NAMES:
                    target_dir = folder / dir_name
                    if target_dir.exists() and target_dir.is_dir():
                        target_dirs.append(target_dir)

            # 查找检查文件夹直接子文件
            if DELETE_FILES:
                for file_name in TARGET_FILE_NAMES:
                    target_file = folder / file_name
                    if target_file.exists() and target_file.is_file():
                        target_files.append(target_file)

    return exam_folders, target_dirs, target_files


def delete_targets(raw_root: Path, dry_run: bool = True):
    if not raw_root.exists():
        raise FileNotFoundError(f"RAW_ROOT 不存在: {raw_root}")

    exam_folders, target_dirs, target_files = collect_targets(raw_root)

    print("=" * 100)
    print(f"RAW_ROOT: {raw_root}")
    print(f"找到检查文件夹数量: {len(exam_folders)}")
    print(f"找到待删除文件夹数量: {len(target_dirs)}")
    print(f"找到待删除文件数量: {len(target_files)}")
    print(f"当前模式: {'预览模式，不会删除' if dry_run else '删除模式，会真正删除'}")
    print("=" * 100)

    if len(target_dirs) == 0 and len(target_files) == 0:
        print("没有找到需要删除的文件夹或文件。")
        return

    if len(target_dirs) > 0:
        print("\n待删除文件夹：")
        print("-" * 100)

        for idx, target_dir in enumerate(target_dirs, start=1):
            print(f"[DIR {idx}/{len(target_dirs)}] {target_dir}")

            if not dry_run:
                shutil.rmtree(
                    target_dir,
                    onerror=handle_remove_readonly
                )

    if len(target_files) > 0:
        print("\n待删除文件：")
        print("-" * 100)

        for idx, target_file in enumerate(target_files, start=1):
            print(f"[FILE {idx}/{len(target_files)}] {target_file}")

            if not dry_run:
                remove_readonly_file(target_file)

    print("=" * 100)

    if dry_run:
        print("预览完成。确认路径无误后，将 DRY_RUN 改为 False 再运行。")
    else:
        print("删除完成。")


delete_targets(RAW_ROOT, dry_run=DRY_RUN)

In [ ]:
## 删除每个case下的_match_logs文件夹

from pathlib import Path
import shutil
import os
import stat


# =========================
# 1. 修改成你的 raw 根目录
# =========================
RAW_ROOT = Path(r"G:\wxy_codes\Fetal_hydronephrosis\dataset\2第二批数据及处理过程\batch2data_5\raw")


# =========================
# 2. 是否只预览
# True：只打印将要删除的文件夹，不真正删除
# False：真正删除
# =========================
DRY_RUN = False


# =========================
# 3. 要删除的文件夹名称
# =========================
TARGET_DIR_NAME = "_match_logs"


def is_case_folder(folder: Path) -> bool:
    """
    判断是否为 case 文件夹。
    例如：
    case_0013
    case_0014
    case_0015
    """
    if not folder.is_dir():
        return False

    return folder.name.startswith("case_")


def handle_remove_readonly(func, path, exc_info):
    """
    处理 Windows 下只读文件导致无法删除的问题。
    """
    os.chmod(path, stat.S_IWRITE)
    func(path)


def delete_match_logs(raw_root: Path, dry_run: bool = True):
    if not raw_root.exists():
        raise FileNotFoundError(f"RAW_ROOT 不存在: {raw_root}")

    case_folders = []
    target_dirs = []

    # 只遍历 raw 根目录下的直接子文件夹
    for case_folder in raw_root.iterdir():
        if is_case_folder(case_folder):
            case_folders.append(case_folder)

            match_logs_dir = case_folder / TARGET_DIR_NAME

            if match_logs_dir.exists() and match_logs_dir.is_dir():
                target_dirs.append(match_logs_dir)

    print("=" * 100)
    print(f"RAW_ROOT: {raw_root}")
    print(f"找到 case 文件夹数量: {len(case_folders)}")
    print(f"找到待删除 _match_logs 文件夹数量: {len(target_dirs)}")
    print(f"当前模式: {'预览模式，不会删除' if dry_run else '删除模式，会真正删除'}")
    print("=" * 100)

    if len(target_dirs) == 0:
        print("没有找到需要删除的 _match_logs 文件夹。")
        return

    for idx, target_dir in enumerate(target_dirs, start=1):
        print(f"[{idx}/{len(target_dirs)}] {target_dir}")

        if not dry_run:
            shutil.rmtree(
                target_dir,
                onerror=handle_remove_readonly
            )

    print("=" * 100)

    if dry_run:
        print("预览完成。确认路径无误后，将 DRY_RUN 改为 False 再运行。")
    else:
        print("删除完成。")


delete_match_logs(RAW_ROOT, dry_run=DRY_RUN)